V10 Мат модель для нахождения коэффициента влияния гендера на ЗП

In [ ]:
import pandas as pd

modelv1 = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

model v10 (линейная регрессия по параметрам: salary, gender, experience, education_type, region_code, industry_code)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

modelv1 = modelv1[
    (modelv1['salary'] > 5000) &
    (modelv1['salary'] < 200000) &
    (modelv1['experience_mistake'] != 1) &
    (modelv1['birthday_mistake'] != 1) &
    (modelv1['gender'].isin(['Мужской', 'Женский'])) &
    (modelv1['industry_code'] == 'BuldindRealty') &
    (modelv1['region_code'].notna())
].copy()

modelv1['male'] = (modelv1['gender'] == 'Мужской').astype(int)
modelv1['experience'] = modelv1['experience'].fillna(0)

results = []

for code, grp in modelv1.groupby('region_code'):
    nm = (grp['gender'] == 'Мужской').sum()
    nf = (grp['gender'] == 'Женский').sum()
    if nm < 5 or nf < 5:
        continue

    modelv1 = smf.ols('salary ~ male + experience', data=grp).fit()

    results.append({
        'регион': int(code),
        'n_М': nm,
        'n_Ж': nf,
        'n_всего': len(grp),
        'коэф_gender': round(modelv1.params['male'], 0),
        'p_gender': round(modelv1.pvalues['male'], 4),
        'gender_значимо': 'да' if modelv1.pvalues['male'] < 0.05 else 'нет',
        'коэф_опыт': round(modelv1.params['experience'], 0),
        'p_опыт': round(modelv1.pvalues['experience'], 4),
        'R2': round(modelv1.rsquared, 3)
    })

out = pd.DataFrame(results).sort_values('p_gender')
print(out.to_string(index=False))
out.to_excel('../../data/processed/v11_result.xlsx', index=False)